# Monoforest: один запрос
Загрузите JAR, `linear_monomials.json` и папку готового индекса на сервер, затем укажите пути ниже.
Нужна Java 11+. Запускается один запрос без прогрева и повторов.
`search_ms` включает поиск и получение текстов документов; загрузка индекса показана отдельно.


In [ ]:
from pathlib import Path
import json
import subprocess

JAVA = "/usr/lib/jvm/java-1.11.0-openjdk-amd64/bin/java"  # Java 11 из вашего JH
HEAP = "4g"

JAR = Path("monoforest-0.1.0-SNAPSHOT-runner.jar")
INDEX = Path("/home/anglukhikhan/unpacked/lucene_index_positions")
MODEL = Path("linear_monomials.json")
QUERY = "grammar terms medicine"
TOP_K = 10
THRESHOLD = 3.0
RAM_ROOT = "/run/jupyter-monitor/monoforest-anglukhikhan"  # tmpfs вашего JH; None — исходный индекс


In [ ]:
if RAM_ROOT is not None:
    Path(RAM_ROOT).mkdir(mode=0o700, exist_ok=True)

command = [
    JAVA, f"-Xmx{HEAP}", "-jar", str(JAR),
    "--index", str(INDEX), "--model", str(MODEL), "--query", QUERY,
    "--top-k", str(TOP_K), "--threshold", str(THRESHOLD),
]
if RAM_ROOT is not None:
    command += ["--ram-root", RAM_ROOT]

print("Загрузка индекса и выполнение одного запроса…")
completed = subprocess.run(command, capture_output=True, text=True)
if completed.returncode:
    raise RuntimeError(completed.stderr or completed.stdout)
response = json.loads(completed.stdout)

print(f"Копирование в RAM: {response['copy_to_ram_ms']:.3f} мс")
print(f"Открытие модели и индекса: {response['open_model_and_index_ms']:.3f} мс")
print(f"Поиск с получением документов: {response['search_ms']:.3f} мс")
print(f"Документов возвращено: {response['result']['returned']}")
for rank, doc in enumerate(response["result"]["candidates"], 1):
    print(f"\n{rank}. {doc['doc_id']} | score={doc['score']}")
    print(doc["title"] if doc["title"] is not None else "(title не сохранён в индексе)")
    print(doc["text"] if doc["text"] is not None else "(text не сохранён в индексе)")


Выбранный tmpfs должен вмещать индекс, а доступной памяти JupyterHub должно хватать на индекс, JVM и резерв.
RAM-копия удаляется после запроса. tmpfs может использовать swap; строгое нахождение в RAM требует
отключённого swap либо tmpfs с `noswap`. Повторный запуск ячейки заново копирует и открывает индекс.
